# Completion Outcome Reanalysis

This notebook reprocesses the saved `full_transcript` request artifacts from the 12 completed runs and computes more informative `completion_outcome` metrics directly in notebook cell outputs.

The analysis is retrospective only: it does not make new API calls and it does not export CSV tables.

In [1]:
from pathlib import Path
import json
import re
import math
from collections import Counter
import statistics

import pandas as pd

repo = Path.cwd().resolve().parent if Path.cwd().name == 'code' else Path.cwd().resolve()
scenario_dir = repo / 'data' / 'scenarios' / 'synthetic_firefighter_radio_controlled_v1'
outputs_dir = repo / 'code' / 'outputs'
batch_dir = outputs_dir / 'final_eval_20260321__batch'
batch_tables_dir = batch_dir / 'tables'
run_dirs = sorted(outputs_dir.glob('final_eval_20260321__r*'))

print(f'repo: {repo}')
print(f'scenarios: {scenario_dir}')
print(f'runs found: {len(run_dirs)}')
print(f'batch tables dir: {batch_tables_dir}')

repo: D:\VSCWorkSpace\swisstext26
scenarios: D:\VSCWorkSpace\swisstext26\data\scenarios\synthetic_firefighter_radio_controlled_v1
runs found: 12
batch tables dir: D:\VSCWorkSpace\swisstext26\code\outputs\final_eval_20260321__batch\tables


## Method

The notebook computes four retrospective comparisons between predicted and gold `completion_outcome` values:

- `exact`: the current repo behavior, i.e. normalized exact string match.
- `exact_stripped`: exact match after removing routine radio boilerplate such as `An Einsatzleitung von ...`, `Meldung:`, and `Schluss`.
- `token_f1`: token-overlap F1 on boilerplate-stripped text.
- `rouge_l_f1`: LCS-based ROUGE-L F1 on boilerplate-stripped text.

The main interpretation focuses on `gold_completed=True`, because the current dataset release stores non-null `completion_outcome` strings even for some incomplete tasks.

In [2]:
def normalize_text(text: str | None) -> str | None:
    if text is None:
        return None
    return re.sub(r'\s+', ' ', text.lower().strip())


def strip_radio_boilerplate(text: str | None) -> str | None:
    if text is None:
        return None
    text = normalize_text(text)
    text = re.sub(r'^an einsatzleitung von [^,]+, meldung:\s*', '', text)
    text = re.sub(r'^an einsatzleitung von [^,]+:\s*', '', text)
    text = re.sub(r'^meldung:\s*', '', text)
    text = re.sub(r',?\s*schluss$', '', text)
    text = re.sub(r'\s+', ' ', text).strip(' ;,.')
    return text or None


def tokenize(text: str | None) -> list[str]:
    if not text:
        return []
    return re.findall(r'[a-z0-9]+', text.lower())


def token_f1(pred_text: str | None, gold_text: str | None) -> float:
    pred_tokens = tokenize(pred_text)
    gold_tokens = tokenize(gold_text)
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    pred_counts = Counter(pred_tokens)
    gold_counts = Counter(gold_tokens)
    overlap = sum((pred_counts & gold_counts).values())
    if overlap == 0:
        return 0.0
    precision = overlap / sum(pred_counts.values())
    recall = overlap / sum(gold_counts.values())
    return 2 * precision * recall / (precision + recall)


def rouge_l_f1(pred_text: str | None, gold_text: str | None) -> float:
    pred_tokens = tokenize(pred_text)
    gold_tokens = tokenize(gold_text)
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    dp = [0] * (len(gold_tokens) + 1)
    for pred_token in pred_tokens:
        prev = 0
        for j, gold_token in enumerate(gold_tokens, start=1):
            cur = dp[j]
            if pred_token == gold_token:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = cur
    lcs = dp[-1]
    if lcs == 0:
        return 0.0
    precision = lcs / len(pred_tokens)
    recall = lcs / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def load_gold_map() -> dict[str, dict[str, dict]]:
    gold_map: dict[str, dict[str, dict]] = {}
    for scenario_path in sorted(scenario_dir.glob('*.json')):
        scenario = json.loads(scenario_path.read_text(encoding='utf-8'))
        gold_map[scenario['scenario_id']] = {row['task_id']: row for row in scenario['gold_task_states']}
    return gold_map


def collect_rows() -> pd.DataFrame:
    gold_map = load_gold_map()
    rows: list[dict] = []
    for request_path in sorted(outputs_dir.glob('final_eval_20260321__r*/requests/*/*/full_transcript/full.json')):
        run_id = request_path.parts[-6]
        scenario_id = request_path.parts[-4]
        structure_condition = request_path.parts[-3]
        request_obj = json.loads(request_path.read_text(encoding='utf-8'))
        predicted_states = {row['task_id']: row for row in request_obj['parsed_output']['task_states']}
        for task_id, gold in gold_map[scenario_id].items():
            pred = predicted_states[task_id]
            gold_outcome = gold.get('completion_outcome')
            pred_outcome = pred.get('completion_outcome')
            gold_norm = normalize_text(gold_outcome)
            pred_norm = normalize_text(pred_outcome)
            gold_stripped = strip_radio_boilerplate(gold_outcome)
            pred_stripped = strip_radio_boilerplate(pred_outcome)
            rows.append(
                {
                    'run_id': run_id,
                    'scenario_id': scenario_id,
                    'structure_condition': structure_condition,
                    'task_id': task_id,
                    'gold_completed': bool(gold['completed']),
                    'pred_completed': bool(pred['completed']),
                    'gold_outcome': gold_outcome,
                    'pred_outcome': pred_outcome,
                    'gold_stripped': gold_stripped,
                    'pred_stripped': pred_stripped,
                    'exact': float((gold_norm == pred_norm) and (gold_outcome is not None or pred_outcome is not None)),
                    'exact_stripped': float((gold_stripped == pred_stripped) and (gold_outcome is not None or pred_outcome is not None)),
                    'token_f1': token_f1(pred_stripped, gold_stripped),
                    'rouge_l_f1': rouge_l_f1(pred_stripped, gold_stripped),
                }
            )
    return pd.DataFrame(rows)


def mean_std_ci95(values: list[float]) -> tuple[float, float, float]:
    if not values:
        return 0.0, 0.0, 0.0
    mean_value = statistics.mean(values)
    std_value = statistics.pstdev(values)
    ci95 = 1.96 * std_value / math.sqrt(len(values)) if len(values) > 0 else 0.0
    return mean_value, std_value, ci95


In [3]:
df = collect_rows()
print(f'task-level rows: {len(df)}')
print(f'runs: {df.run_id.nunique()}')
print(f'scenarios: {df.scenario_id.nunique()}')
print(f'structure conditions: {df.structure_condition.nunique()}')
print('\nGold completion split:')
print(df['gold_completed'].value_counts().rename(index={True: 'completed', False: 'incomplete'}).to_string())
print('\nRows with non-null gold outcome:', int(df['gold_outcome'].notna().sum()))
print('Rows with non-null predicted outcome:', int(df['pred_outcome'].notna().sum()))

task-level rows: 540
runs: 12
scenarios: 5
structure conditions: 3

Gold completion split:
gold_completed
completed     432
incomplete    108

Rows with non-null gold outcome: 540
Rows with non-null predicted outcome: 468


In [4]:
metric_cols = ['exact', 'exact_stripped', 'token_f1', 'rouge_l_f1']

subsets = {
    'all_rows': df,
    'gold_completed_only': df[df['gold_completed']],
    'gold_incomplete_only': df[~df['gold_completed']],
}

summary_rows = []
for subset_name, subset in subsets.items():
    row = {'subset': subset_name, 'n': int(len(subset))}
    for metric in metric_cols:
        row[metric] = subset[metric].mean()
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df

,subset,n,exact,exact_stripped,token_f1,rouge_l_f1
0,all_rows,540,0.0,0.087037,0.592157,0.560907
1,gold_completed_only,432,0.0,0.108796,0.674265,0.640020
2,gold_incomplete_only,108,0.0,0.000000,0.263722,0.244456


In [5]:
completed_df = df[df['gold_completed']].copy()
structure_summary = (
    completed_df.groupby('structure_condition', as_index=False)[metric_cols]
    .mean()
    .sort_values('structure_condition')
    .reset_index(drop=True)
)
structure_summary

,structure_condition,exact,exact_stripped,token_f1,rouge_l_f1
0,continuous_transcript,0.0,0.125000,0.669305,0.637973
1,no_speaker,0.0,0.118056,0.689433,0.656710
2,structured_dialogue,0.0,0.083333,0.664058,0.625377


In [6]:
run_summary = (
    completed_df.groupby('run_id', as_index=False)[['token_f1', 'rouge_l_f1']]
    .mean()
    .sort_values('run_id')
    .reset_index(drop=True)
)
print('Completed-only run mean summary:')
print(f"token_f1 mean={run_summary['token_f1'].mean():.4f} std={statistics.pstdev(run_summary['token_f1']):.4f}")
print(f"rouge_l_f1 mean={run_summary['rouge_l_f1'].mean():.4f} std={statistics.pstdev(run_summary['rouge_l_f1']):.4f}")
run_summary

Completed-only run mean summary:
token_f1 mean=0.6743 std=0.0143
rouge_l_f1 mean=0.6400 std=0.0154


,run_id,token_f1,rouge_l_f1
0,final_eval_20260321__r01,0.655916,0.621510
1,final_eval_20260321__r02,0.656540,0.625695
2,final_eval_20260321__r03,0.662823,0.624561
3,final_eval_20260321__r04,0.676070,0.644057
4,final_eval_20260321__r05,0.678741,0.639947
5,final_eval_20260321__r06,0.665650,0.627204
6,final_eval_20260321__r07,0.691408,0.654446
7,final_eval_20260321__r08,0.663339,0.630148
8,final_eval_20260321__r09,0.697515,0.665811
9,final_eval_20260321__r10,0.664823,0.628985


In [7]:
example_cols = [
    'run_id',
    'scenario_id',
    'structure_condition',
    'task_id',
    'token_f1',
    'gold_outcome',
    'pred_outcome',
]

lowest_examples = completed_df.sort_values(['token_f1', 'run_id']).head(5)[example_cols]
highest_examples = completed_df.sort_values(['token_f1', 'run_id'], ascending=[False, True]).head(5)[example_cols]

print('Lowest-overlap examples among completed tasks:')
display(lowest_examples)
print('\nHighest-overlap examples among completed tasks:')
display(highest_examples)

Lowest-overlap examples among completed tasks:


,run_id,scenario_id,structure_condition,task_id,token_f1,gold_outcome,pred_outcome
1,final_eval_20260321__r01,S1,continuous_transcript,T2,0.32,"An Einsatzleitung von Angriffstrupp, Meldung: ...","Wohnung 2. OG links abgesucht, keine Personen ..."
4,final_eval_20260321__r01,S1,no_speaker,T2,0.32,"An Einsatzleitung von Angriffstrupp, Meldung: ...","Wohnung 2. OG links abgesucht, keine Personen ..."
7,final_eval_20260321__r01,S1,structured_dialogue,T2,0.32,"An Einsatzleitung von Angriffstrupp, Meldung: ...","Wohnung 2. OG links abgesucht, keine Personen ..."
46,final_eval_20260321__r02,S1,continuous_transcript,T2,0.32,"An Einsatzleitung von Angriffstrupp, Meldung: ...","Wohnung 2. OG links abgesucht, keine Personen ..."
49,final_eval_20260321__r02,S1,no_speaker,T2,0.32,"An Einsatzleitung von Angriffstrupp, Meldung: ...","Wohnung 2. OG links abgesucht, keine Personen ..."



Highest-overlap examples among completed tasks:


,run_id,scenario_id,structure_condition,task_id,token_f1,gold_outcome,pred_outcome
0,final_eval_20260321__r01,S1,continuous_transcript,T1,1.0,"An Einsatzleitung von Wassertrupp, Meldung: Wa...",Wasserversorgung ab Hydrant Nord steht bis Hau...
3,final_eval_20260321__r01,S1,no_speaker,T1,1.0,"An Einsatzleitung von Wassertrupp, Meldung: Wa...",Wasserversorgung ab Hydrant Nord bis Hauseinga...
6,final_eval_20260321__r01,S1,structured_dialogue,T1,1.0,"An Einsatzleitung von Wassertrupp, Meldung: Wa...",Wasserversorgung ab Hydrant Nord bis Hauseinga...
9,final_eval_20260321__r01,S2,continuous_transcript,T1,1.0,"An Einsatzleitung von Sicherheitstrupp, Meldun...","Tiefgarage stromlos, Garagentor auf Handbetrie..."
12,final_eval_20260321__r01,S2,no_speaker,T1,1.0,"An Einsatzleitung von Sicherheitstrupp, Meldun...","Tiefgarage stromlos, Garagentor auf Handbetrie..."


## Batch Export

The next cell writes one minimal batch-level CSV summary with one row per transcript structure condition. Each row summarizes per-run means for `gold_completed_only`, matching the batch reporting style used elsewhere in the project.


In [8]:
batch_rows = []
subset_name = 'gold_completed_only'
for structure_condition, subset_df in completed_df.groupby('structure_condition', sort=True):
    per_run = (
        subset_df.groupby('run_id', as_index=False)[metric_cols]
        .mean()
        .sort_values('run_id')
        .reset_index(drop=True)
    )
    batch_row = {
        'subset': subset_name,
        'structure_condition': structure_condition,
        'aggregation': 'per_condition',
        'n_runs': int(per_run['run_id'].nunique()),
        'n_task_rows_total': int(len(subset_df)),
        'n_task_rows_per_run': float(len(subset_df) / per_run['run_id'].nunique()),
    }
    for metric in metric_cols:
        mean_value, std_value, ci95_value = mean_std_ci95(per_run[metric].tolist())
        batch_row[f'{metric}_mean'] = mean_value
        batch_row[f'{metric}_std'] = std_value
        batch_row[f'{metric}_ci95'] = ci95_value
    batch_rows.append(batch_row)
batch_export_df = pd.DataFrame(batch_rows)
batch_export_df = batch_export_df.sort_values('structure_condition').reset_index(drop=True)
batch_tables_dir.mkdir(parents=True, exist_ok=True)
export_path = batch_tables_dir / 'completion_outcome_reanalysis_summary.csv'
batch_export_df.to_csv(export_path, index=False)
print(f'Wrote {export_path}')
batch_export_df


Wrote D:\VSCWorkSpace\swisstext26\code\outputs\final_eval_20260321__batch\tables\completion_outcome_reanalysis_summary.csv


,subset,structure_condition,aggregation,n_runs,n_task_rows_total,n_task_rows_per_run,exact_mean,exact_std,exact_ci95,exact_stripped_mean,exact_stripped_std,exact_stripped_ci95,token_f1_mean,token_f1_std,token_f1_ci95,rouge_l_f1_mean,rouge_l_f1_std,rouge_l_f1_ci95
0,gold_completed_only,continuous_transcript,per_condition,12,144,12.0,0.0,0.0,0.0,0.125000,0.041667,0.023575,0.669305,0.027217,0.015400,0.637973,0.029304,0.016580
1,gold_completed_only,no_speaker,per_condition,12,144,12.0,0.0,0.0,0.0,0.118056,0.063267,0.035797,0.689433,0.022623,0.012800,0.656710,0.025668,0.014523
2,gold_completed_only,structured_dialogue,per_condition,12,144,12.0,0.0,0.0,0.0,0.083333,0.000000,0.000000,0.664058,0.013547,0.007665,0.625377,0.012757,0.007218


## Interpretation

If this notebook reproduces `exact = 0.0` while giving clearly positive `token_f1` and `rouge_l_f1` values on completed tasks, then the original zero is best interpreted as a metric mismatch: the model is often paraphrasing the completion evidence instead of reproducing the exact gold message string.